## ⚖️ Extension 2: Counterfactual Fairness

What this Extension Task is Used For

Counterfactual Fairness tests whether changing a protected attribute (such as gender) changes the model's decision while keeping everything else identical.

The goal is to answer:

"Would this applicant receive a different decision if only their gender changed?"

A fair model should ideally produce the same decision.

In [32]:

import pandas as pd
from typing import TypedDict

from sklearn.preprocessing import LabelEncoder

from langgraph.graph import StateGraph, END

# State Definition

class FairnessState(TypedDict):
    df_orig: pd.DataFrame
    clf: object
    scaler: object
    feature_cols_all: list

    X_orig: pd.DataFrame
    X_cf: pd.DataFrame

    pred_orig: list
    pred_cf: list

    pct_changed: float
    report: str

# Node 1: Encode Data

def encode_features(state: FairnessState):

    df_test = state["df_orig"].copy()

    for col in ["gender", "region", "employment_type"]:
        le = LabelEncoder()
        df_test[col + "_enc"] = le.fit_transform(df_test[col])

    X_orig = df_test[state["feature_cols_all"]].copy()

    return {
        "df_orig": df_test,
        "X_orig": X_orig
    }

# Node 2: Create Counterfactual

def create_counterfactual(state: FairnessState):

    X_cf = state["X_orig"].copy()

    # flip gender
    X_cf["gender_enc"] = 1 - X_cf["gender_enc"]

    return {"X_cf": X_cf}

# Node 3: Predict

def predict_decisions(state: FairnessState):

    clf = state["clf"]
    scaler = state["scaler"]

    pred_orig = clf.predict(
        scaler.transform(state["X_orig"])
    )

    pred_cf = clf.predict(
        scaler.transform(state["X_cf"])
    )

    return {
        "pred_orig": pred_orig,
        "pred_cf": pred_cf
    }

# Node 4: Fairness Evaluation

def evaluate_fairness(state: FairnessState):

    pred_orig = state["pred_orig"]
    pred_cf = state["pred_cf"]

    n_changed = (pred_orig != pred_cf).sum()
    pct_changed = n_changed / len(pred_orig)

    return {
        "pct_changed": pct_changed
    }


# Node 5: Report Generation

def generate_report(state: FairnessState):

    df_test = state["df_orig"].copy()

    df_test["orig_pred"] = state["pred_orig"]
    df_test["cf_pred"] = state["pred_cf"]

    changed = df_test[
        state["pred_orig"] != state["pred_cf"]
    ][
        [
            "gender",
            "region",
            "credit_score",
            "income",
            "orig_pred",
            "cf_pred"
        ]
    ]

    verdict = (
        "❌ NOT counterfactually fair — gender influences decisions"
        if state["pct_changed"] > 0.05
        else
        "✅ Approximately counterfactually fair"
    )

    report = f"""
⚖️ COUNTERFACTUAL FAIRNESS TEST — Gender

Total applicants: {len(df_test)}
Decision changed on flip: {(state['pred_orig'] != state['pred_cf']).sum()}
Percentage changed: {state['pct_changed']:.2%}

Verdict:
{verdict}

Sample Changed Decisions:
{changed.head(10).to_string()}
"""

    print(report)

    return {"report": report}

In [29]:
# Build LangGraph

graph = StateGraph(FairnessState)

graph.add_node("encode", encode_features)
graph.add_node("counterfactual", create_counterfactual)
graph.add_node("predict", predict_decisions)
graph.add_node("evaluate", evaluate_fairness)
graph.add_node("report", generate_report)

graph.set_entry_point("encode")

graph.add_edge("encode", "counterfactual")
graph.add_edge("counterfactual", "predict")
graph.add_edge("predict", "evaluate")
graph.add_edge("evaluate", "report")
graph.add_edge("report", END)

app = graph.compile()

In [31]:
# Excute the Graph
result = app.invoke(
    {
        "df_orig": df,
        "clf": clf,
        "scaler": scaler,
        "feature_cols_all": all_features
    }
)

print(result["pct_changed"])


⚖️ COUNTERFACTUAL FAIRNESS TEST — Gender

Total applicants: 5
Decision changed on flip: 0
Percentage changed: 0.00%

Verdict:
✅ Approximately counterfactually fair

Sample Changed Decisions:
Empty DataFrame
Columns: [gender, region, credit_score, income, orig_pred, cf_pred]
Index: []

0.0


Sample Flipped Decisions Table

This table shows applicants whose decisions changed after only gender was modified.

Interpretation:

If many decisions flip, gender is influencing model behavior.

A high percentage indicates a fairness issue.

Why LangGraph Helps

| Node           | Responsibility                         |
| -------------- | -------------------------------------- |
| encode         | Label encoding and feature preparation |
| counterfactual | Create gender-flipped applicants       |
| predict        | Run model inference                    |
| evaluate       | Compute fairness metric                |
| report         | Generate explainable output            |
